In [3]:
# Using pizza,steak,susxhi, 10 and 20 percent dartaset

# dowanload data
from download_data import download_data

data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                              destination="pizza_steak_sushi_20_percent")

data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                              destination="pizza_steak_sushi")


Unzipping file..
Unzipping file..


In [6]:
# train dir path
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"
# test psth

test_dir = data_10_percent_path/"test"

# lets check the paths
train_dir_10_percent, train_dir_20_percent , test_dir


(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi_20_percent/train'),
 PosixPath('data/pizza_steak_sushi/test'))

In [11]:
# Turn the data set into data loader
from going_modular.going_modular import data_setup
from torchvision import transforms

# setup the image transform as the pretrained model expects
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

manual_transform = transforms.Compose([
    transforms.Resize(size=(244,244)),
    transforms.ToTensor(),
    normalize
])


# lets creat ethe data loaders
train_data_loader_10_percent , test_data_loader, class_names = data_setup.create_dataloaders(train_dir=train_dir_10_percent, 
                                                                                test_dir=test_dir,
                                                                                transform=manual_transform,
                                                                                batch_size=32
                                                                                )

train_data_loader_20_percent , test_data_loader, class_names = data_setup.create_dataloaders(train_dir=train_dir_20_percent, 
                                                                                test_dir=test_dir,
                                                                                transform=manual_transform,
                                                                                batch_size=32
                                                                                )

print(f"Len of train data 10% : {len(train_data_loader_10_percent)} and train data 20% : {len(train_data_loader_20_percent)}")
print(f"Test data loader len : {len(test_data_loader)}")
print(f"Class names : {class_names}")

Len of train data 10% : 8 and train data 20% : 15
Test data loader len : 3
Class names : ['pizza', 'steak', 'sushi']


In [14]:
# creat eth pretrained model
from torchvision.models import EfficientNet_B0_Weights,  EfficientNet_B2_Weights
from torchvision.models import efficientnet_b0, efficientnet_b2
EffNetB0 = efficientnet_b0(weights = EfficientNet_B0_Weights.DEFAULT)
EffNetB2 = efficientnet_b2(weights = EfficientNet_B2_Weights.DEFAULT)

In [25]:
# get summary of model
from torchinfo import summary

summary(model=EffNetB0,
        input_size=(32,3,244,244),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"],
        verbose=0)

Layer (type (var_name))                                      Input Shape               Output Shape              Param #                   Trainable
EfficientNet (EfficientNet)                                  [32, 3, 244, 244]         [32, 1000]                --                        True
├─Sequential (features)                                      [32, 3, 244, 244]         [32, 1280, 8, 8]          --                        True
│    └─Conv2dNormActivation (0)                              [32, 3, 244, 244]         [32, 32, 122, 122]        --                        True
│    │    └─Conv2d (0)                                       [32, 3, 244, 244]         [32, 32, 122, 122]        864                       True
│    │    └─BatchNorm2d (1)                                  [32, 32, 122, 122]        [32, 32, 122, 122]        64                        True
│    │    └─SiLU (2)                                         [32, 32, 122, 122]        [32, 32, 122, 122]        --                

In [26]:

summary(model=EffNetB2,
        input_size=(32,3,244,244),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"],
        verbose=0)

Layer (type (var_name))                                      Input Shape               Output Shape              Param #                   Trainable
EfficientNet (EfficientNet)                                  [32, 3, 244, 244]         [32, 1000]                --                        True
├─Sequential (features)                                      [32, 3, 244, 244]         [32, 1408, 8, 8]          --                        True
│    └─Conv2dNormActivation (0)                              [32, 3, 244, 244]         [32, 32, 122, 122]        --                        True
│    │    └─Conv2d (0)                                       [32, 3, 244, 244]         [32, 32, 122, 122]        864                       True
│    │    └─BatchNorm2d (1)                                  [32, 32, 122, 122]        [32, 32, 122, 122]        64                        True
│    │    └─SiLU (2)                                         [32, 32, 122, 122]        [32, 32, 122, 122]        --                

In [51]:
# lets functionize the freeze paramter action
OUT_FEATURES = len(class_names)
import torch
from torch import nn

def create_effnetb0():
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)
    
    for param in model.features.parameters():
        param.requires_grad = False
    
    torch.manual_seed(42)
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features=1280, out_features=OUT_FEATURES, bias=True)
    ).to(device="cpu")
    
    model.name = "effnetb0"
    
    return model
    
    
    
def create_effnetb2():
    weights = EfficientNet_B2_Weights.DEFAULT
    model = efficientnet_b2(weights=weights)
    
    for param in model.features.parameters():
        param.requires_grad = False
    
    torch.manual_seed(42)
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=OUT_FEATURES, bias=True)
    ).to(device="cpu")
    
    model.name = "effnetb2"
    
    return model
    

In [50]:
m1 = create_effnetb0()
m2 = create_effnetb2()


m2.classifier





Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1408, out_features=3, bias=True)
)

In [52]:
summary(model=m1,
        input_size=(32,3,244,244),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"],
        verbose=0)

Layer (type (var_name))                                      Input Shape               Output Shape              Param #                   Trainable
EfficientNet (EfficientNet)                                  [32, 3, 244, 244]         [32, 3]                   --                        Partial
├─Sequential (features)                                      [32, 3, 244, 244]         [32, 1280, 8, 8]          --                        False
│    └─Conv2dNormActivation (0)                              [32, 3, 244, 244]         [32, 32, 122, 122]        --                        False
│    │    └─Conv2d (0)                                       [32, 3, 244, 244]         [32, 32, 122, 122]        (864)                     False
│    │    └─BatchNorm2d (1)                                  [32, 32, 122, 122]        [32, 32, 122, 122]        (64)                      False
│    │    └─SiLU (2)                                         [32, 32, 122, 122]        [32, 32, 122, 122]        --         

In [53]:
summary(model=m2,
        input_size=(32,3,244,244),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"],
        verbose=0)

Layer (type (var_name))                                      Input Shape               Output Shape              Param #                   Trainable
EfficientNet (EfficientNet)                                  [32, 3, 244, 244]         [32, 3]                   --                        Partial
├─Sequential (features)                                      [32, 3, 244, 244]         [32, 1408, 8, 8]          --                        False
│    └─Conv2dNormActivation (0)                              [32, 3, 244, 244]         [32, 32, 122, 122]        --                        False
│    │    └─Conv2d (0)                                       [32, 3, 244, 244]         [32, 32, 122, 122]        (864)                     False
│    │    └─BatchNorm2d (1)                                  [32, 32, 122, 122]        [32, 32, 122, 122]        (64)                      False
│    │    └─SiLU (2)                                         [32, 32, 122, 122]        [32, 32, 122, 122]        --         

In [54]:
num_epochs = [5,10]
models = ["effnetb0", "effnetb2"]

train_data_loaders = {"data_10_percent" : train_data_loader_10_percent,
                      "data_20_percent" : train_data_loader_20_percent}



In [60]:
from going_modular.going_modular import utils
from train_update import train
from writer import create_writer


torch.manual_seed(42)

exp_num = 0 

for dataloader_name, train_data_loader in train_data_loaders.items():
    for epoch in num_epochs:
        for model_name in models:
            
            exp_num += 1
            print(f"[INFO] Experiment Number : {exp_num}")
            print(f"[INFO] Model : {model_name}")
            print(f"[INFO] Data loader : {dataloader_name}")
            print(f"[INFO] Number of epochs : {epoch}")
            
            if model_name == "effnetb0":
                model = create_effnetb0()
            else:
                model = create_effnetb2()
                
            loss_fn = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            
            train(model=model,
                  train_dataloader=train_data_loader,
                  test_dataloader=test_data_loader,
                  loss_fn=loss_fn,
                  optimizer=optimizer,
                  epochs=epoch,
                  device="cpu",
                  writer=create_writer(exp_name=dataloader_name,
                                       model_name=model_name,
                                       extra=f"{epoch}_epoch"))
            
            # save modle
            
            save_file_path = f"07_{model_name}_{dataloader_name}_{epoch}_epochs.pt"
            utils.save_model(model=model, target_dir="models", model_name=save_file_path)
            print("-"*50 + "\n")
            



            
    

[INFO] Experiment Number : 1
[INFO] Model : effnetb0
[INFO] Data loader : data_10_percent
[INFO] Number of epochs : 5
[INFO] : Cretaed summary writer and save to logs


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0425 | train_acc: 0.4727 | test_loss: 0.9077 | test_acc: 0.5189


 20%|██        | 1/5 [01:30<06:02, 90.56s/it]

Epoch: 2 | train_loss: 0.8667 | train_acc: 0.6953 | test_loss: 0.7919 | test_acc: 0.7216


...

Epoch: 9 | train_loss: 0.3265 | train_acc: 0.9313 | test_loss: 0.3455 | test_acc: 0.9384


 90%|█████████ | 9/10 [16:52<01:53, 113.08s/it]

Epoch: 10 | train_loss: 0.2786 | train_acc: 0.9354 | test_loss: 0.3483 | test_acc: 0.9384


100%|██████████| 10/10 [18:44<00:00, 112.42s/it]

[INFO] Saving model to: models/07_effnetb2_data_20_percent_10_epochs.pt
--------------------------------------------------



In [4]:
%load_ext tensorboard

In [6]:
%tensorboard --logdir runs